In [3]:
!pip install numpy

   ---------------------------------------- 0.0/16.3 MB ? eta -:--:--
   ----- ---------------------------------- 2.4/16.3 MB 10.3 MB/s eta 0:00:02
   ----------- ---------------------------- 4.7/16.3 MB 11.0 MB/s eta 0:00:02
   ------------------ --------------------- 7.3/16.3 MB 11.3 MB/s eta 0:00:01
   ------------------------ --------------- 10.0/16.3 MB 11.5 MB/s eta 0:00:01
   ------------------------------ --------- 12.6/16.3 MB 11.6 MB/s eta 0:00:01
   ------------------------------------ --- 14.9/16.3 MB 11.6 MB/s eta 0:00:01
   ---------------------------------------- 16.3/16.3 MB 11.0 MB/s eta 0:00:00


In [20]:
import numpy as np
import os

def load_npy_files(directory):
    npy_files = [f for f in os.listdir(directory) if f.endswith('metric_depth.npy')]
    data_list = []
    for file in npy_files:
        data = np.load(os.path.join(directory, file))
        data_list.append(data)
    return data_list

directory_path = 'content/depth'
depth_list = load_npy_files(directory_path)

# 로드된 데이터 확인
# for i, data in enumerate(depth_list):
#     print(f"Data {i+1}: {data}")
print(len(depth_list))


180


In [21]:
fx, fy, cx, cy = 800, 800, 640, 360
height, width = depth_list[0].shape

points_3d = [[] for _ in range(len(depth_list))]
for i, depth in enumerate(depth_list):
    for v in range(height):
        for u in range(width):
            d = depth[v, u]
            if d == 0:
                continue
            Z = d
            X = (u - cx) * Z / fx
            Y = (v - cy) * Z / fy
            points_3d[i].append((X, Y, Z))

    points_3d[i] = np.array(points_3d[i])

In [22]:
print(points_3d[0])

[[-64.         -36.          80.        ]
 [-63.9        -36.          80.        ]
 [-63.8        -36.          80.        ]
 ...
 [  8.48463379   4.78176378  10.65574102]
 [ 10.19028715   5.73403305  12.77778954]
 [ 10.74101712   6.03446815  13.44728278]]


In [9]:
!pip install opencv-python

   ---------------------------------------- 0.0/38.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/38.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/38.8 MB ? eta -:--:--
    --------------------------------------- 0.5/38.8 MB 1.5 MB/s eta 0:00:26
   - -------------------------------------- 1.0/38.8 MB 1.9 MB/s eta 0:00:20
   -- ------------------------------------- 2.4/38.8 MB 3.4 MB/s eta 0:00:11
   ----- ---------------------------------- 5.0/38.8 MB 5.5 MB/s eta 0:00:07
   ------- -------------------------------- 7.6/38.8 MB 6.9 MB/s eta 0:00:05
   -------- ------------------------------- 8.4/38.8 MB 7.1 MB/s eta 0:00:05
   -------- ------------------------------- 8.4/38.8 MB 7.1 MB/s eta 0:00:05
   --------- ------------------------------ 9.7/38.8 MB 5.8 MB/s eta 0:00:06
   ---------- ----------------------------- 10.2/38.8 MB 5.5 MB/s eta 0:00:06
   ----------- ---------------------------- 10.7/38.8 MB 5.1 MB/s eta 0:00:06
   ----------- ---

In [10]:
import cv2
cap = cv2.VideoCapture('content/test.mp4')
fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()

In [33]:
print(fps)

15.0


In [49]:
print(height, width)

720 1280


In [46]:
tracked_data = np.load('content/tracker/tracked_data.npy', allow_pickle=True)
center_points = [[] for _ in range(len(tracked_data))]
for i, data in enumerate(tracked_data):
    if 'vehicle1' in data:
        points = data['vehicle1']
        center = int((points[0]+points[2])/2), int((points[1]+points[3])/2)
        center_points[i].append(center)
    else:
        center = None, None
        center_points[i].append(center)

In [47]:
print(center_points[0])

[(959, 495)]


In [44]:
print(points_3d[0].shape)

(921600, 3)


In [55]:
# 특정 포인트의 이동 거리 구하기
velocity = []
for i, center in enumerate(center_points):
    if center[0][0] is None:
        continue
    else:
        v, u = center[0][0], center[0][1]
        point_idx = int(v * height + u)

        next = i+1
        while next <= len(center_points)-2 and center_points[next][0][0] is None:

            next+=1
        v, u = center_points[next][0][0], center_points[next][0][1]
        if v is None:
            continue
        next_idx = int(v * height + u)
        
        delx = np.linalg.norm(np.array(points_3d[i][point_idx]) - np.array(points_3d[next][next_idx]))
        velocity.append(delx*fps*(next-i))
        # x/(frame*#f) * frame/s * #f 

In [56]:
print(velocity)

[np.float64(262.6741789402443), np.float64(254.31267508202882), np.float64(134.26968991408907), np.float64(254.3207551007943), np.float64(93.72209732545016), np.float64(293.2547874637204), np.float64(76.2730929848269), np.float64(36.367858038282655), np.float64(125.08383925697262), np.float64(454.8650087912035), np.float64(46.81721221596557), np.float64(198.66286141058305), np.float64(364.64332360719027), np.float64(83.45437352696443), np.float64(225.9884487525358), np.float64(276.25603078054644), np.float64(347.99734057183844), np.float64(354.73835982772766), np.float64(295.4929367914295), np.float64(34.138973146972035), np.float64(387.6541804915577), np.float64(107.98184732963043), np.float64(285.5686075139806), np.float64(8.598913768803495), np.float64(287.8036131894149), np.float64(278.78875485746227), np.float64(270.45211694499454), np.float64(10.473758420027028), np.float64(309.79183043503997), np.float64(16.68676482688609), np.float64(1.131661194745647), np.float64(128.617468964

In [58]:
len(velocity)

81

In [67]:
# 특정 포인트의 이동 거리 구하기
velocity = []
for i, center in enumerate(center_points):
    if center[0][0] is None:
        velocity.append(None)
        continue
    else:
        v, u = center[0][0], center[0][1]
        point_idx = int(v * height + u)

        next = i+1
        while next <= len(center_points)-2 and center_points[next][0][0] is None:
            next+=1
            
        v, u = center_points[next][0][0], center_points[next][0][1]
        if v is None:
            velocity.append(None)
            continue
        next_idx = int(v * height + u)
        delx = np.linalg.norm(np.array(points_3d[i][point_idx]) - np.array(points_3d[next][next_idx]))
        velocity.append(delx*fps*(next-i))
        # x/(frame*#f) * frame/s * #f 

In [68]:

print(len(velocity))

150
